# Recruitment Predictor

## Run Order
1. Step 0: Setup + Load Data  
2. Step 1: Data Preparation  
3. Step 2: EDA + Hypothesis Testing  
4. Step 3: ML Pipeline  
5. Step 4: Multicollinearity (VIF)  
6. Step 5: Class Imbalance Handling  
7. Step 6: Finalization (save best model)

This notebook is arranged to run top-to-bottom without dependency issues.


## Step 0: Setup + Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind, chi2_contingency

from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

sns.set_theme(style='whitegrid')
alpha = 0.05

In [ ]:
df = pd.read_csv('recruitment_data.csv')
print('Shape:', df.shape)
display(df.head())

print('\nDtypes:')
display(df.dtypes)

print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False))

## Step 1: Data Preparation

In [ ]:
df_prep = df.copy()

# 1) Missing values
num_cols = df_prep.select_dtypes(include=['number']).columns
cat_cols = df_prep.select_dtypes(include=['object', 'category']).columns

for c in num_cols:
    df_prep[c] = df_prep[c].fillna(df_prep[c].median())
for c in cat_cols:
    mode_val = df_prep[c].mode(dropna=True)
    fill_val = mode_val.iloc[0] if not mode_val.empty else 'Unknown'
    df_prep[c] = df_prep[c].fillna(fill_val)

# 2) Encode categorical features (only if text)
encoders = {}
for c in ['Gender', 'EducationLevel']:
    if c in df_prep.columns and str(df_prep[c].dtype) in ['object', 'category']:
        le = LabelEncoder()
        df_prep[c] = le.fit_transform(df_prep[c].astype(str))
        encoders[c] = dict(zip(le.classes_, le.transform(le.classes_)))

# 3) Target label
if 'HiringDecision' in df_prep.columns:
    y = df_prep['HiringDecision'].astype(int)
else:
    score = 0.5 * df_prep['InterviewScore'] + 0.3 * df_prep['SkillScore'] + 0.2 * df_prep['PersonalityScore']
    df_prep['HiringDecision'] = (score >= score.median()).astype(int)
    y = df_prep['HiringDecision']

# 4) Derived features
df_prep['ExperienceBucket'] = pd.cut(
    df_prep['ExperienceYears'],
    bins=[-1, 2, 5, 10, np.inf],
    labels=['Entry', 'Junior', 'Mid', 'Senior']
)
df_prep['ExperienceBucketCode'] = df_prep['ExperienceBucket'].cat.codes

# Stage-style scores
df_prep['TechnicalStageScore'] = 0.6 * df_prep['SkillScore'] + 0.4 * df_prep['InterviewScore']
df_prep['BehavioralStageScore'] = 0.7 * df_prep['PersonalityScore'] + 0.3 * df_prep['InterviewScore']
df_prep['OverallCompositeScore'] = (
    0.4 * df_prep['InterviewScore'] +
    0.35 * df_prep['SkillScore'] +
    0.25 * df_prep['PersonalityScore']
)

print('Prepared shape:', df_prep.shape)
print('Target distribution:')
display(df_prep['HiringDecision'].value_counts().sort_index())
display(df_prep[['ExperienceYears','ExperienceBucket','ExperienceBucketCode','TechnicalStageScore','BehavioralStageScore','OverallCompositeScore']].head())

## Step 2: EDA + Hypothesis Testing

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_prep, x='HiringDecision')
plt.title('HiringDecision Distribution')
plt.show()

num_features = [
    'Age', 'ExperienceYears', 'PreviousCompanies', 'DistanceFromCompany',
    'InterviewScore', 'SkillScore', 'PersonalityScore',
    'TechnicalStageScore', 'BehavioralStageScore', 'OverallCompositeScore'
]
for col in num_features:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df_prep, x='HiringDecision', y=col)
    plt.title(f'{col} by HiringDecision')
    plt.show()

In [ ]:
corr = df_prep.select_dtypes(include='number').corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

print('Top correlations with HiringDecision:')
display(corr['HiringDecision'].sort_values(ascending=False))

In [ ]:
# Hypothesis tests
results = []

# t-test: InterviewScore by target
g0 = df_prep.loc[df_prep['HiringDecision'] == 0, 'InterviewScore']
g1 = df_prep.loc[df_prep['HiringDecision'] == 1, 'InterviewScore']
t_stat, p1 = ttest_ind(g0, g1, equal_var=False)
results.append(['InterviewScore mean diff (t-test)', p1, p1 < alpha])

# chi-square: EducationLevel vs target
ct_edu = pd.crosstab(df_prep['EducationLevel'], df_prep['HiringDecision'])
chi2_edu, p2, _, _ = chi2_contingency(ct_edu)
results.append(['EducationLevel vs HiringDecision (chi-square)', p2, p2 < alpha])

# chi-square: ExperienceBucket vs target
ct_exp = pd.crosstab(df_prep['ExperienceBucket'], df_prep['HiringDecision'])
chi2_exp, p3, _, _ = chi2_contingency(ct_exp)
results.append(['ExperienceBucket vs HiringDecision (chi-square)', p3, p3 < alpha])

print('Education contingency:')
display(ct_edu)
print('Experience contingency:')
display(ct_exp)

display(pd.DataFrame(results, columns=['Test','p_value','significant_at_0.05']))

## Step 3: ML Pipeline

In [ ]:
X = df_prep.drop(columns=['HiringDecision'])
y = df_prep['HiringDecision'].astype(int)

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
def evaluate_model(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    out = {
        'model': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall': recall_score(y_te, y_pred, zero_division=0),
        'f1': f1_score(y_te, y_pred, zero_division=0),
        'roc_auc': np.nan
    }
    if hasattr(model, 'predict_proba'):
        out['roc_auc'] = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    return out

In [ ]:
# Logistic baseline
log_reg = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=3000, penalty='l2', C=1e6, solver='lbfgs'))
])
log_reg.fit(X_train, y_train)
metrics_lr = evaluate_model('Logistic Regression (baseline)', log_reg, X_test, y_test)

# Logistic tuned (L1/L2)
log_reg_tuned = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=3000, solver='liblinear'))
])
param_grid_lr = {'model__penalty': ['l1', 'l2'], 'model__C': [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(log_reg_tuned, param_grid_lr, cv=5, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train, y_train)
metrics_lr_reg = evaluate_model('Logistic Regression (L1/L2 tuned)', grid_lr.best_estimator_, X_test, y_test)

# Decision tree tuned + pruning
tree_pipe = Pipeline([
    ('prep', preprocessor_tree),
    ('model', DecisionTreeClassifier(random_state=42))
])
param_grid_tree = {
    'model__max_depth': [3, 5, 8, 12, None],
    'model__min_samples_leaf': [1, 5, 10],
    'model__ccp_alpha': [0.0, 0.001, 0.005, 0.01]
}
grid_tree = GridSearchCV(tree_pipe, param_grid_tree, cv=5, scoring='f1', n_jobs=-1)
grid_tree.fit(X_train, y_train)
metrics_tree = evaluate_model('Decision Tree (tuned + pruning)', grid_tree.best_estimator_, X_test, y_test)

# KNN tuned
knn_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', KNeighborsClassifier())
])
param_grid_knn = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['minkowski'],
    'model__p': [1, 2]
}
grid_knn = GridSearchCV(knn_pipe, param_grid_knn, cv=5, scoring='f1', n_jobs=-1)
grid_knn.fit(X_train, y_train)
metrics_knn = evaluate_model('KNN (tuned)', grid_knn.best_estimator_, X_test, y_test)

results_df = pd.DataFrame([metrics_tree, metrics_lr, metrics_lr_reg, metrics_knn]).sort_values('f1', ascending=False)
display(results_df.style.format({'accuracy':'{:.3f}','precision':'{:.3f}','recall':'{:.3f}','f1':'{:.3f}','roc_auc':'{:.3f}'}))

In [ ]:
# Bias-variance diagnostics
# Validation curve for tree depth
depth_range = [1,2,3,4,5,6,8,10,12,15]
tr, va = validation_curve(
    grid_tree.best_estimator_, X_train, y_train,
    param_name='model__max_depth', param_range=depth_range,
    cv=5, scoring='f1', n_jobs=-1
)

plt.figure(figsize=(8,5))
plt.plot(depth_range, tr.mean(axis=1), marker='o', label='Train F1')
plt.plot(depth_range, va.mean(axis=1), marker='s', label='Validation F1')
plt.xlabel('max_depth')
plt.ylabel('F1')
plt.title('Bias-Variance Trend (Decision Tree)')
plt.legend()
plt.show()

# Learning curve for tuned logistic
sizes, tr_lc, va_lc = learning_curve(
    grid_lr.best_estimator_, X_train, y_train,
    train_sizes=np.linspace(0.1,1.0,6), cv=5, scoring='f1', n_jobs=-1
)
plt.figure(figsize=(8,5))
plt.plot(sizes, tr_lc.mean(axis=1), marker='o', label='Train F1')
plt.plot(sizes, va_lc.mean(axis=1), marker='s', label='Validation F1')
plt.xlabel('Training examples')
plt.ylabel('F1')
plt.title('Learning Curve (Logistic Tuned)')
plt.legend()
plt.show()

## Step 4: Multicollinearity (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

def calculate_vif(df_numeric):
    x = df_numeric.replace([np.inf, -np.inf], np.nan).dropna().copy()
    x_const = sm.add_constant(x, has_constant='add')
    return pd.DataFrame({
        'feature': x.columns,
        'VIF': [variance_inflation_factor(x_const.values, i+1) for i in range(len(x.columns))]
    }).sort_values('VIF', ascending=False)

X_full = df_prep.drop(columns=['HiringDecision']).copy()
num_cols_vif = X_full.select_dtypes(include=['number']).columns.tolist()
cat_cols_vif = X_full.select_dtypes(include=['object','category']).columns.tolist()

vif_initial = calculate_vif(X_full[num_cols_vif])
print('Initial VIF:')
display(vif_initial)

selected_num = num_cols_vif.copy()
removed = []
threshold = 10.0
while len(selected_num) > 2:
    vif_now = calculate_vif(X_full[selected_num])
    if float(vif_now.iloc[0]['VIF']) <= threshold:
        break
    f = str(vif_now.iloc[0]['feature'])
    removed.append((f, float(vif_now.iloc[0]['VIF'])))
    selected_num.remove(f)

print('Removed features:')
display(pd.DataFrame(removed, columns=['feature_removed','vif_at_removal']))
print('Final VIF:')
display(calculate_vif(X_full[selected_num]))

In [ ]:
# Retrain logistic on VIF-reduced features
final_features = selected_num + cat_cols_vif
X_vif = X_full[final_features].copy()
y_vif = df_prep['HiringDecision'].astype(int)

X_train_vif, X_test_vif, y_train_vif, y_test_vif = train_test_split(
    X_vif, y_vif, test_size=0.2, random_state=42, stratify=y_vif
)

num_vif = X_vif.select_dtypes(include=['number']).columns.tolist()
cat_vif = X_vif.select_dtypes(include=['object','category']).columns.tolist()

preprocessor_vif = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_vif),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_vif)
])

log_reg_vif = Pipeline([
    ('prep', preprocessor_vif),
    ('model', LogisticRegression(max_iter=3000, solver='liblinear'))
])

grid_lr_vif = GridSearchCV(
    log_reg_vif,
    {'model__penalty':['l1','l2'], 'model__C':[0.01,0.1,1,10,100]},
    cv=5, scoring='f1', n_jobs=-1
)
grid_lr_vif.fit(X_train_vif, y_train_vif)
metrics_lr_vif = evaluate_model('Logistic Regression (VIF-reduced features)', grid_lr_vif.best_estimator_, X_test_vif, y_test_vif)

display(pd.DataFrame([metrics_lr_reg, metrics_lr_vif]).set_index('model').style.format('{:.3f}'))

## Step 5: Class Imbalance Handling

In [ ]:
print('Class counts:')
display(y.value_counts().sort_index())
print('Class ratio:')
display((y.value_counts().sort_index()/len(y)).round(3))

# Logistic with class_weight='balanced'
log_reg_bal = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=3000, solver='liblinear', class_weight='balanced'))
])
grid_lr_bal = GridSearchCV(
    log_reg_bal,
    {'model__penalty':['l1','l2'], 'model__C':[0.01,0.1,1,10,100]},
    cv=5, scoring='f1', n_jobs=-1
)
grid_lr_bal.fit(X_train, y_train)
metrics_lr_bal = evaluate_model('Logistic (balanced class_weight)', grid_lr_bal.best_estimator_, X_test, y_test)

# Tree with class_weight='balanced'
tree_bal = Pipeline([
    ('prep', preprocessor_tree),
    ('model', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])
grid_tree_bal = GridSearchCV(
    tree_bal,
    {'model__max_depth':[3,5,8,12,None], 'model__min_samples_leaf':[1,5,10], 'model__ccp_alpha':[0.0,0.001,0.005,0.01]},
    cv=5, scoring='f1', n_jobs=-1
)
grid_tree_bal.fit(X_train, y_train)
metrics_tree_bal = evaluate_model('Decision Tree (balanced class_weight)', grid_tree_bal.best_estimator_, X_test, y_test)

# Optional SMOTE
smote_results = None
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    smote_lr = ImbPipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', LogisticRegression(max_iter=3000, solver='liblinear'))
    ])
    grid_smote_lr = GridSearchCV(
        smote_lr,
        {'model__penalty':['l1','l2'], 'model__C':[0.01,0.1,1,10,100]},
        cv=5, scoring='f1', n_jobs=-1
    )
    grid_smote_lr.fit(X_train, y_train)
    smote_results = evaluate_model('Logistic (SMOTE)', grid_smote_lr.best_estimator_, X_test, y_test)
except Exception as e:
    print('SMOTE skipped:', e)

rows = [metrics_tree, metrics_lr_reg, metrics_lr_vif, metrics_tree_bal, metrics_lr_bal]
if smote_results is not None:
    rows.append(smote_results)
imbalance_compare = pd.DataFrame(rows).drop_duplicates('model').sort_values('f1', ascending=False)
display(imbalance_compare.style.format({'accuracy':'{:.3f}','precision':'{:.3f}','recall':'{:.3f}','f1':'{:.3f}','roc_auc':'{:.3f}'}))

## Step 6: Finalization

In [ ]:
import joblib
from pathlib import Path
import json

artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(exist_ok=True)

candidates = []
for name, obj, xte, yte in [
    ('grid_tree', grid_tree, X_test, y_test),
    ('grid_lr', grid_lr, X_test, y_test),
    ('grid_knn', grid_knn, X_test, y_test),
    ('grid_tree_bal', grid_tree_bal, X_test, y_test),
    ('grid_lr_bal', grid_lr_bal, X_test, y_test),
]:
    best_est = obj.best_estimator_
    candidates.append((name, best_est, f1_score(yte, best_est.predict(xte), zero_division=0)))

if 'grid_lr_vif' in globals():
    est = grid_lr_vif.best_estimator_
    candidates.append(('grid_lr_vif', est, f1_score(y_test_vif, est.predict(X_test_vif), zero_division=0)))

best_label, final_model, best_f1 = sorted(candidates, key=lambda t: t[2], reverse=True)[0]
joblib.dump(final_model, artifacts_dir / 'best_recruitment_model.joblib')

summary = {'selected_model_label': best_label, 'selection_metric': 'f1', 'test_f1': round(float(best_f1), 6)}
with open(artifacts_dir / 'model_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Saved:', artifacts_dir / 'best_recruitment_model.joblib')
print('Saved:', artifacts_dir / 'model_summary.json')
print(summary)

In [ ]:
# Final model interpretation
model_obj = final_model.named_steps['model']
prep_obj = final_model.named_steps['prep']
feature_names = prep_obj.get_feature_names_out()

if 'DecisionTree' in type(model_obj).__name__:
    imp = pd.Series(model_obj.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
    plt.figure(figsize=(8,6))
    imp.sort_values().plot(kind='barh')
    plt.title('Top 15 Feature Importances (Final Model)')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    display(imp)
else:
    coefs = pd.Series(model_obj.coef_.ravel(), index=feature_names).sort_values(key=np.abs, ascending=False).head(15)
    display(coefs)